# A Tessera database in a notebook, and the map over it

`td.create()` makes the database, `stage` and `declare_*` fill it, `commit()` builds and serves it, and `db.map()` is the explorer over it. The corpus is `data/notebook/`: 50,000 arXiv papers, their k-means clusters and the topics over them.

The database's three planes bind loopback at port 0 and `serve.cors_loopback` admits a page served from a loopback address, so this notebook needs no fixed port and no origin list. The session credential stays in the kernel; what the page gets is a token minted from it.

In [ ]:
import os
import pathlib

import tesseradb as td

DATA = pathlib.Path(os.environ.get("TESSERA_NOTEBOOK_DATA", "../../../data/notebook"))

In [ ]:
db = td.create()  # a temporary directory, on /dev/shm where there is one
db.stage("points", str(DATA / "points.parquet"), default=True)
for name, file in [
    ("kmeans", "clusters-kmeans"),
    ("kmeans_members", "clusters-kmeans-members"),
    ("kmeans_topics", "topics-kmeans"),
    ("kmeans_topic_members", "topics-kmeans-members"),
]:
    db.stage(name, str(DATA / f"{file}.parquet"))

db.declare_view("s0", source="points", access="categories", title="arXiv, 50,000 papers")
db.declare_attribute("title", type="text", index=True)
db.declare_attribute("arxiv_id", type="keyword", index=True, title="arXiv ID")
db.declare_layer(
    "clusters/kmeans",
    kind="flat",
    source="kmeans",
    members="kmeans_members",
    require_member_visibility={"count": 50},
    title="k-means clusters",
)
db.declare_labels(
    "topics/kmeans",
    of="clusters/kmeans",
    source="kmeans_topics",
    members="kmeans_topic_members",
    content_requires="all",
    title="k-means topics",
)
print(db.commit())  # tessera check, tessera build, tessera serve

In [ ]:
m = db.map(colour_by="cluster:clusters/kmeans", height=520)
m

Pan and zoom; pick a point; draw a box (shift-drag) or a lasso. Then read the widget in the next cell.

In [ ]:
m.bbox, m.layers, m.colour_by, m.selected, m.selected_artifact, m.region

Setting a control trait redraws. `filters` takes the wire's expression; `bbox` moves the camera.

In [ ]:
m.colour_by = None
m.filters = None

`db.viewer(terms)` is any principal's map — computed inside their own mask, not filtered down from the operator's. A term the database has staged no label for is refused and named.

In [ ]:
db.viewer(["cs.LG"]).map(height=380)

The query verbs go through the same plane with the same token, never by reading the bundle. `viewport()` is the points served for a box, as a pyarrow table whose schema metadata carries the counts; `item()` is one record.

In [ ]:
points = db.viewport(k=64)
points.column_names, points.num_rows

In [ ]:
db.item(points.column("tessera_id")[0].as_py())["fields"]

A deployment somebody else runs is the same widget and the same read verbs, against a token that deployment issued you. There is no `viewer(terms)` there and no write verb: minting another principal needs the session credential, and writing needs the operator's.

```python
v = td.connect("https://tessera.example/viewer", token=my_token)
v.map(colour_by="cluster:clusters/kmeans")
```

Its page calls the viewer plane from this page's origin, so that origin must be in the deployment's CORS list.

In [ ]:
db.close()  # stops the server; a token it minted stays good until its lifetime runs out